## Training notebook

In [1]:
import gym
import highway_env

# from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
import tensorflow as tf

%load_ext tensorboard
from datetime import datetime
import os
from ipywidgets import interact, widgets
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback, BaseCallback

from typing import Callable

### Select operating system

In [3]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UNIX', 'WINDOWS'])

interactive(children=(Dropdown(description='desired_os', options=('UNIX', 'WINDOWS'), value='UNIX'), Output())…

<function __main__.f(desired_os)>

In [4]:
OUTPUT_PATH = '/Users/fornerispighetti/HighwayDRL/training_output/' if os_id.value == "UNIX" else 'C:/Users/pigo/Desktop/HighwayDRL/training_output/'

### Select environment

In [5]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0','multipleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('dm-env-v0', 'acc-dm-env-v0', 'jam-dm-env-v0'…

<function __main__.f(input_env)>

In [7]:
total_timesteps = 3e4
# Create and wrap the environment
env = gym.make(env_id.value)
env.configure({
    "training_total_timesteps": total_timesteps
})

# Create a TensorboardCallback to plot sub-rewards
class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.not_in_RML_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.not_in_RML_rew += self.training_env.get_attr('not_in_RML_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0]
            self.logger.record("eval/coll_rew", self.coll_rew)
            self.logger.record("episode/not_in_RML_rew", self.not_in_RML_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.not_in_RML_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=False, render=False)

In [8]:
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func

In [10]:
ilr = 3.5e-4
# PPO parameters
model = RecurrentPPO("MlpLstmPolicy",
        env,
        n_steps=128,
        learning_rate=ilr,
        verbose=1,
        batch_size=256,
        seed=1,
        n_epochs=10,
        max_grad_norm=1,
        gae_lambda=0.98,
        policy_kwargs=dict(
            net_arch=[dict(vf=[64])],
            lstm_hidden_size=64,
            ortho_init=False,
            enable_critic_lstm=True,
        ),
        tensorboard_log=f'{OUTPUT_PATH}logdir'
        )

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [11]:
# Train the agent for n steps
model.learn(total_timesteps=int(total_timesteps), callback=[checkpoint_callback, eval_callback, TensorboardCallback()])

# Save the trained agent
model.save(f'{OUTPUT_PATH}models/RECppo_RL_NOlinear_{str(os_id.value)}_{total_timesteps:.1E}_{env_id.value}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Logging to C:/Users/pigo/Desktop/HighwayDRL/training_output/logdir\RecurrentPPO_16
---------------------------------
| episode/           |          |
|    high_speed_rew  | 0.275    |
|    not_in_RML_rew  | -0.273   |
| eval/              |          |
|    coll_rew        | 4        |
| rollout/           |          |
|    ep_len_mean     | 20       |
|    ep_rew_mean     | 15.8     |
| time/              |          |
|    fps             | 27       |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 128      |
---------------------------------
---------------------------------------
| episode/                |           |
|    high_speed_rew       | 0.127     |
|    not_in_RML_rew       | -0.18     |
| eval/                   |           |
|    coll_rew             | 6         |
| rollout/                |           |
|    ep_len_mean          | 36.6      |
|    ep_rew_mean          | 32.3      |
| time/                   |           |
|    fp

c:\Users\pigo\miniconda3\envs\highway\lib\site-packages\stable_baselines3\common\evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=56.76 +/- 46.98
Episode length: 63.40 +/- 50.11
----------------------------------------
| episode/                |            |
|    high_speed_rew       | 0.18       |
|    not_in_RML_rew       | -0.105     |
| eval/                   |            |
|    coll_rew             | 25         |
|    mean_ep_length       | 63.4       |
|    mean_reward          | 56.8       |
| time/                   |            |
|    total_timesteps      | 1000       |
| train/                  |            |
|    approx_kl            | 0.25236338 |
|    clip_fraction        | 0.618      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.418     |
|    explained_variance   | -0.05      |
|    learning_rate        | 0.00035    |
|    loss                 | 28.2       |
|    n_updates            | 70         |
|    policy_gradient_loss | -0.346     |
|    value_loss           | 60.5       |
----------------------------------------
New best m

KeyboardInterrupt: 

### Continued learning

In [3]:
env_cl_id = 'dm-env-v0'
env_cl = gym.make(env_cl_id)

In [8]:
custom_objects = { 'learning_rate': 3.5e-4 }

model_cl = RecurrentPPO.load(f"{OUTPUT_PATH}models/RECppo_RL_NOlinear_WINDOWS_3.0E+04_multipleO-dm-env-v0_07-07_02-26-08", env=env_cl, custom_objects=custom_objects, tensorboard_log=f"{OUTPUT_PATH}logdir")

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [9]:
class TensorboardCallback(BaseCallback):
    def __init__(self, verbose=1):
        super(TensorboardCallback, self).__init__(verbose)
        self.coll_rew = 0
        self.not_in_RML_rew = 0
        self.high_speed_rew = 0
        self.current_steps = 0

    def _on_step(self) -> bool:        
        self.not_in_RML_rew += self.training_env.get_attr('not_in_RML_reward')[0]
        self.high_speed_rew += self.training_env.get_attr('high_speed_reward')[0]
            
        if self.locals["dones"][0]:
            self.coll_rew += -self.training_env.get_attr('collision_reward')[0]
            self.logger.record("eval/coll_rew", self.coll_rew)
            self.logger.record("episode/not_in_RML_rew", self.not_in_RML_rew / self.current_steps)
            self.logger.record("episode/high_speed_rew", self.high_speed_rew / self.current_steps)
            self.not_in_RML_rew = self.high_speed_rew = self.current_steps = 0
        else:
            self.current_steps = self.training_env.get_attr('steps')[0]
        return True

# Save a checkpoint every 1000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}checkpoints',
                                         name_prefix='dm_rl_callback')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=False, render=False)

In [10]:
new_timesteps = 3e4

model_cl.learn(total_timesteps=int(new_timesteps), callback=[checkpoint_callback, eval_callback, TensorboardCallback()])

model_cl.save(f'{OUTPUT_PATH}models/RECppo_RL_{str(os_id.value)}_{new_timesteps:.1E}_{env_cl_id}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Logging to C:/Users/pigo/Desktop/HighwayDRL/training_output/logdir\RecurrentPPO_16
---------------------------------
| episode/           |          |
|    high_speed_rew  | 0.38     |
|    not_in_RML_rew  | -0.219   |
| eval/              |          |
|    coll_rew        | 5        |
| rollout/           |          |
|    ep_len_mean     | 21.8     |
|    ep_rew_mean     | 17.2     |
| time/              |          |
|    fps             | 7        |
|    iterations      | 1        |
|    time_elapsed    | 16       |
|    total_timesteps | 128      |
---------------------------------
---------------------------------------
| episode/                |           |
|    high_speed_rew       | 0.121     |
|    not_in_RML_rew       | -0.3      |
| eval/                   |           |
|    coll_rew             | 7         |
| rollout/                |           |
|    ep_len_mean          | 20.9      |
|    ep_rew_mean          | 16.3      |
| time/                   |           |
|    fp

c:\Users\pigo\miniconda3\envs\highway\lib\site-packages\stable_baselines3\common\evaluation.py:65: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Eval num_timesteps=1000, episode_reward=22.24 +/- 12.62
Episode length: 27.00 +/- 14.70
---------------------------------------
| episode/                |           |
|    high_speed_rew       | 0.297     |
|    not_in_RML_rew       | -0.25     |
| eval/                   |           |
|    coll_rew             | 37        |
|    mean_ep_length       | 27        |
|    mean_reward          | 22.2      |
| time/                   |           |
|    total_timesteps      | 1000      |
| train/                  |           |
|    approx_kl            | 1.872258  |
|    clip_fraction        | 0.585     |
|    clip_range           | 0.2       |
|    entropy_loss         | -0.00308  |
|    explained_variance   | -2.38e-07 |
|    learning_rate        | 0.00035   |
|    loss                 | 119       |
|    n_updates            | 2420      |
|    policy_gradient_loss | 0.262     |
|    value_loss           | 227       |
---------------------------------------
New best mean reward!
----------

KeyboardInterrupt: 

In [ ]:
model_cl.save(f'{OUTPUT_PATH}models/RECppo_RL_{str(os_id.value)}_1e5_DMenv_cl3_{datetime.datetime.today().day}-{datetime.datetime.today().month}-{datetime.datetime.today().year}')